In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import datetime as dt

In [3]:
start = dt.datetime(2013,6,1)
end = dt.datetime(2022,2,11)

stk_data = yf.download("RELIANCE.NS", start=start, end=end)

[*********************100%***********************]  1 of 1 completed


In [4]:
stk_data

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2013-06-03,166.953690,170.651287,166.392150,170.301652,14165128
2013-06-04,165.322052,169.475242,165.099566,167.525789,14752690
2013-06-05,169.941406,170.428770,165.385609,165.385609,12748842
2013-06-06,167.822449,170.344022,166.996060,169.517633,17113393
2013-06-07,166.042526,169.941434,165.491595,168.288627,9420701
...,...,...,...,...,...
2022-02-04,1060.769409,1068.572936,1056.128310,1065.183039,11061241
2022-02-07,1054.308472,1072.372440,1048.802746,1065.638259,10714467


In [5]:
# The dataframe is not normal columns, it’s a MultiIndex (two-level columns).
# So, flattening the columns:

stk_data.columns = stk_data.columns.get_level_values(0)

In [6]:
stk_data=stk_data[["Open", "High", "Low", "Close"]]

In [7]:
stk_data

Price,Open,High,Low,Close
Date,,,,
2013-06-03,170.301652,170.651287,166.392150,166.953690
2013-06-04,167.525789,169.475242,165.099566,165.322052
2013-06-05,165.385609,170.428770,165.385609,169.941406
2013-06-06,169.517633,170.344022,166.996060,167.822449
2013-06-07,168.288627,169.941434,165.491595,166.042526
...,...,...,...,...
2022-02-04,1065.183039,1068.572936,1056.128310,1060.769409
2022-02-07,1065.638259,1072.372440,1048.802746,1054.308472
2022-02-08,1060.473750,1073.828376,1051.077695,1072.031006


In [8]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1 = Ms.fit_transform(stk_data)
print("Len:", data1.shape)

Len: (2144, 4)


In [9]:
data1 = pd.DataFrame(data1, columns=["Open", "High", "Low", "Close"])

### Train - Test Split

In [10]:
training_size = round(len(data1) * 0.80)
print(training_size)

X_train = data1[:training_size]
X_test = data1[training_size:]
print("X_train length:", X_train.shape)
print("X_test length:", X_test.shape)

y_train = data1[:training_size]
y_test = data1[training_size:]
print("y_train length:", y_train.shape)
print("y_test length:", y_test.shape)

1715
X_train length: (1715, 4)
X_test length: (429, 4)
y_train length: (1715, 4)
y_test length: (429, 4)


### Model Creation

#### VARMA Model

In [11]:
performance = {"Model": [], "RMSE": [], "MaPe": [], "Lag": [], "Test": []}

In [12]:
def combination_varmax(dataset, list1):

    print("Running for:", list1) # show which variables we are using (e.g., Close, High)

    datasetTwo = dataset[list1]  # select only chosen columns (multivariate input for model)

    test_obs = 28                      # number of future observations to predict (last 28 days)
    train = datasetTwo[:-test_obs]     # training data (all data except last 28 rows)
    test = datasetTwo[-test_obs:]      # testing data (last 28 rows to compare predictions)

    from statsmodels.tsa.statespace.varmax import VARMAX  # import VARMAX model (multivariate AR + MA)

    best_aic = float("inf")  # initialize best AIC as very large number (so any model will be smaller)
    best_order = None        # to store best (p,q) combination
    best_model = None        # to store best fitted model

    # STEP 1: Find best (p,q) - Try different (p,q) combinations
    for p in range(1,4):           # p = number of lag observations (AR part)
        for q in range(0,3):       # q = number of error terms (MA part)
            try:
                model = VARMAX(train, order=(p,q))  # create VARMAX model with given (p,q)
                result = model.fit(disp=False)      # train the model (disp=False → hide iteration output)

                print("Order (p,q):", (p,q), "AIC:", result.aic)  # print model order and its AIC score

                if result.aic < best_aic:  # check if current model is better (lower AIC)
                    best_aic = result.aic  # update best AIC
                    best_order = (p,q)     # store best (p,q)
                    best_model = result    # store best model

            except:
                continue  # skip errors (some combinations may fail)

    print("Best Order Selected:", best_order)  # display best (p,q) based on lowest AIC

    # STEP 2: Use best model
    result = best_model   # use best model for forecasting

    # STEP 3: Forecast
    pred = result.forecast(steps=test_obs)  # forecast next 28 days

    # STEP 4: Evaluation
    import numpy as np
    from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
 
    mse = mean_squared_error(test, pred)  # calculate mean squared error between actual and predicted
    rmse = round(np.sqrt(mse), 3)         # convert MSE to RMSE (real error scale)

    mape = mean_absolute_percentage_error(test, pred)  # calculate percentage error

    # STEP 5: Store results
    performance["Model"].append(list1)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append(best_order)
    performance["Test"].append(test_obs)

    perf = pd.DataFrame(performance)

    return perf, result, pred

In [ ]:
combination_varmax(data1, ["Close", "High"])
combination_varmax(data1, ["Close", "High", "Open"])
combination_varmax(data1, ["Close", "High", "Open", "Low"])

Running for: ['Close', 'High']
Order (p,q): (1, 0) AIC: -19519.791306109924


C:\Users\Default.DESKTOP-73FCCDO\Documents\Installations\envs\aiml\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\Default.DESKTOP-73FCCDO\Documents\Installations\envs\aiml\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Order (p,q): (1, 1) AIC: -17191.379944540473


C:\Users\Default.DESKTOP-73FCCDO\Documents\Installations\envs\aiml\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'


In [ ]:
perf = pd.DataFrame(performance)
perf